# Análisis de perfiles fluviales

Revisamos los perfiles, delimitamos tramos, evaluamos la sensibilidad
a θ y ajustamos relaciones χ–elevación.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import cumulative_trapezoid
from scipy.stats import linregress
from IPython.display import display


# ── Carpeta de entrada ───────────────────────────────────────────────────
BASE = Path(
    r"C:\datos\Memoria"
    r"\Codigos Stanford\Productos"
)

CARPETA_ENTRADA = BASE / "info_perfiles" / "20260929"
archivo_manifest = CARPETA_ENTRADA / "manifest.json"

if not archivo_manifest.is_file():
    raise FileNotFoundError(
        f"No se encontró:\n{archivo_manifest}\n\n"
        "Ejecuta la celda de exportación del notebook anterior "
        "usando esta misma carpeta."
    )


# ── Leer nombres y parámetros ────────────────────────────────────────────
manifest = json.loads(
    archivo_manifest.read_text(encoding="utf-8")
)

if (
    manifest["unidad_distancia"] != "m"
    or manifest["unidad_area"] != "m2"
):
    raise ValueError("Se requiere distancia en metros y área en m².")

nombres = {
    river_id: datos["nombre"]
    for river_id, datos in manifest["rios"].items()
}


# ── Cargar perfiles ──────────────────────────────────────────────────────
perfiles_originales = {}

columnas_requeridas = {
    "distancia_m",
    "elevacion_m",
    "area_m2",
    "x_utm",
    "y_utm",
}

for river_id, datos in manifest["rios"].items():
    archivo = CARPETA_ENTRADA / datos["archivo"]

    if not archivo.is_file():
        raise FileNotFoundError(f"No se encontró el perfil:\n{archivo}")

    p = pd.read_csv(archivo)

    faltantes = columnas_requeridas - set(p.columns)
    if faltantes:
        raise ValueError(
            f"Río {river_id}: faltan columnas {faltantes}"
        )

    p.index.name = "indice_original"
    perfiles_originales[river_id] = p


# ── Configuración gráfica ────────────────────────────────────────────────
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "font.size": 11,
    "savefig.dpi": 300,
})


# ── Resumen de carga ─────────────────────────────────────────────────────
print(f"Entrada utilizada:\n{CARPETA_ENTRADA}")
print(f"\nRíos cargados: {len(perfiles_originales)}")

#display(pd.DataFrame([
#   {
#      "ID": river_id,
#        "rio": nombres[river_id],
#        "theta_anterior": datos["parametros"].get("theta_ref"),
#        "ventana_anterior_m": datos["parametros"].get("ventana_m"),
#    }
#    for river_id, datos in manifest["rios"].items()
#]))

## 1. Delimitación de tramos y revisión de datos

Conservamos los datos originales y definimos intervalos de análisis
mediante distancia desde el outlet.

In [ ]:
# ── MODIFICAR AQUÍ: intervalos que quieres analizar ──────────────────────
# Formato: "ID": [("nombre del tramo", inicio_km, fin_km), ...]

TRAMOS = {
    # "6": [
    #     ("inferior", 2.0, 8.0),
    #     ("superior", 8.0, None),
    # ],
}

# Los intervalos son [inicio, fin): el extremo final no se duplica.
# Para excluir una zona intermedia, deja un espacio entre intervalos.

ids_desconocidos = set(TRAMOS) - set(perfiles_originales)
if ids_desconocidos:
    raise ValueError(f"IDs desconocidos en TRAMOS: {ids_desconocidos}")

perfiles = {}
mascaras_tramos = {}
registros_qc = []

for river_id, original in perfiles_originales.items():
    p = original.copy(deep=True)

    x = p["distancia_m"].to_numpy(dtype=float)
    z = p["elevacion_m"].to_numpy(dtype=float)
    a = p["area_m2"].to_numpy(dtype=float)

    # No integrar sobre distancias o áreas inválidas
    if len(p) < 2 or not np.all(np.isfinite(x)):
        raise ValueError(f"{river_id}: distancia insuficiente o inválida.")

    if np.any(np.diff(x) <= 0):
        raise ValueError(f"{river_id}: la distancia no aumenta estrictamente.")

    if not np.isclose(x[0], 0):
        raise ValueError(f"{river_id}: el perfil no comienza en el outlet.")

    if np.any(~np.isfinite(a) | (a <= 0)):
        raise ValueError(
            f"{river_id}: hay áreas inválidas. Revisar sin borrar las filas."
        )

    p["elevacion_valida"] = np.isfinite(z)

    # Registrar inversiones o zonas planas, sin corregirlas ni usar abs(S)
    pendiente_entre_puntos = np.concatenate((
        [np.nan], np.diff(z) / np.diff(x)
    ))
    p["pendiente_paso"] = pendiente_entre_puntos
    p["paso_no_positivo"] = (
        np.isfinite(pendiente_entre_puntos)
        & (pendiente_entre_puntos <= 0)
    )

    intervalos = TRAMOS.get(
        river_id, [("completo_provisional", 0.0, None)]
    )

    mascaras_tramos[river_id] = {}
    asignado = np.zeros(len(p), dtype=bool)

    for nombre_tramo, inicio_km, fin_km in intervalos:
        if nombre_tramo in mascaras_tramos[river_id]:
            raise ValueError(f"{river_id}: nombre de tramo repetido.")

        inicio = float(inicio_km) * 1000
        fin = np.inf if fin_km is None else float(fin_km) * 1000

        if inicio < 0 or fin <= inicio:
            raise ValueError(f"{river_id}: límites inválidos.")

        mask = (x >= inicio) & (x < fin)

        if not mask.any():
            raise ValueError(f"{river_id} — {nombre_tramo}: tramo vacío.")

        if np.any(mask & asignado):
            raise ValueError(f"{river_id}: hay tramos superpuestos.")

        mascaras_tramos[river_id][nombre_tramo] = mask
        asignado |= mask

    p["incluido_en_tramos"] = asignado
    perfiles[river_id] = p

    registros_qc.append({
        "ID": river_id,
        "rio": nombres[river_id],
        "longitud_km": x[-1] / 1000,
        "puntos": len(p),
        "sin_elevacion": int((~np.isfinite(z)).sum()),
        "pasos_no_positivos": int(p["paso_no_positivo"].sum()),
        "fuera_de_tramos": int((~asignado).sum()),
        "delimitacion": (
            "manual" if river_id in TRAMOS else "completa_provisional"
        ),
    })

control_calidad = pd.DataFrame(registros_qc)

display(control_calidad)

### Visualizar

In [ ]:
# ── MODIFICAR AQUÍ: solo controla las figuras ────────────────────────────
RIOS_VER = ["6"]

# RIOS_VER = ["1", "3", "16"]
# RIOS_VER = list(perfiles)

for river_id in map(str, RIOS_VER):
    if river_id not in perfiles:
        raise ValueError(f"No está cargado el río {river_id}.")

    p = perfiles[river_id]
    fig, ax = plt.subplots(figsize=(10, 5))

    ax.plot(
        p["distancia_m"] / 1000,
        p["elevacion_m"],
        color="0.75",
        label="Perfil original",
    )

    for tramo, mask in mascaras_tramos[river_id].items():
        ax.plot(
            p.loc[mask, "distancia_m"] / 1000,
            p.loc[mask, "elevacion_m"],
            linewidth=2,
            label=tramo,
        )

    ax.set(
        title=f"{river_id} — {nombres[river_id]}",
        xlabel="Distancia desde el outlet (km)",
        ylabel="Elevación (m)",
    )
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 2. Sensibilidad a θ

Recalculamos χ para distintas concavidades y evaluamos la linealidad
de cada tramo. El máximo R² es un diagnóstico, no una validación de θ.

In [ ]:
# ── Parámetros del análisis ─────────────────────────────────────────────
THETAS = np.arange(30, 71, 5) / 100   # Incluye 0.30 y 0.70
THETA_AJUSTE = 0.45                  # Referencia inicial, ajustable
A0_ANALISIS = 1e6                    # m²

# Criterios iniciales para evitar ajustes demasiado cortos
MIN_PUNTOS_AJUSTE = 10
MIN_LONGITUD_M = 500.0

theta_calculados = sorted(set([
    *np.round(THETAS, 8),
    round(THETA_AJUSTE, 8),
]))

chi_sensibilidad = {}
filas_sensibilidad = []
comparacion_chi = []


def ajustar_recta(chi, z, distancia, mask):
    """Ajustar sin eliminar ni modificar filas de la tabla original."""
    valid = mask & np.isfinite(chi) & np.isfinite(z)
    n = int(valid.sum())

    if n < MIN_PUNTOS_AJUSTE:
        return {"estado": "pocos_puntos", "n": n}

    extension = float(np.ptp(distancia[valid]))
    if extension < MIN_LONGITUD_M:
        return {"estado": "tramo_corto", "n": n}

    if np.ptp(chi[valid]) == 0 or np.ptp(z[valid]) == 0:
        return {"estado": "sin_variacion", "n": n}

    fit = linregress(chi[valid], z[valid])
    residuos = z[valid] - (fit.intercept + fit.slope * chi[valid])

    return {
        "estado": "ok" if fit.slope > 0 else "pendiente_no_positiva",
        "n": n,
        "M_chi": float(fit.slope),
        "intercepto_m": float(fit.intercept),
        "R2": float(fit.rvalue**2),
        "RMSE_m": float(np.sqrt(np.mean(residuos**2))),
        "chi_inicio": float(chi[valid].min()),
        "chi_fin": float(chi[valid].max()),
    }


for river_id, p in perfiles.items():
    x = p["distancia_m"].to_numpy(dtype=float)
    a = p["area_m2"].to_numpy(dtype=float)
    z = p["elevacion_m"].to_numpy(dtype=float)

    chi_sensibilidad[river_id] = {}

    for theta in theta_calculados:
        chi = cumulative_trapezoid(
            (A0_ANALISIS / a)**theta,
            x=x,
            initial=0.0,
        )
        chi_sensibilidad[river_id][theta] = chi

        for tramo, mask in mascaras_tramos[river_id].items():
            ajuste = ajustar_recta(chi, z, x, mask)

            filas_sensibilidad.append({
                "ID": river_id,
                "rio": nombres[river_id],
                "tramo": tramo,
                "theta": theta,
                **ajuste,
            })

    # Comparación con χ previo usando SUS parámetros de referencia
    anteriores = manifest["rios"][river_id]["parametros"]

    if (
        "chi_m" in p
        and "theta_ref" in anteriores
        and "A0_m2" in anteriores
    ):
        chi_control = cumulative_trapezoid(
            (anteriores["A0_m2"] / a)**anteriores["theta_ref"],
            x=x,
            initial=0.0,
        )
        chi_previo = p["chi_m"].to_numpy(dtype=float)
        valid = np.isfinite(chi_previo)

        comparacion_chi.append({
            "ID": river_id,
            "diferencia_max_chi_m": (
                float(np.max(np.abs(
                    chi_control[valid] - chi_previo[valid]
                ))) if valid.any() else np.nan
            ),
        })

tabla_sensibilidad = pd.DataFrame(filas_sensibilidad)
#display(tabla_sensibilidad)

print("Comparación de discretizaciones de χ; no sustituye una validación:")
display(pd.DataFrame(comparacion_chi))

In [ ]:
# ── MODIFICAR AQUÍ ──────────────────────────────────────────────────────
RIOS_VER = ["6"]

# RIOS_VER = ["1", "3", "16"]
# RIOS_VER = list(perfiles)

for river_id in map(str, RIOS_VER):
    if river_id not in perfiles:
        raise ValueError(f"No está cargado el río {river_id}.")

    datos = tabla_sensibilidad[
        tabla_sensibilidad["ID"].eq(river_id)
    ]

    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

    for tramo, grupo in datos.groupby("tramo", sort=False):
        grupo = grupo.sort_values("theta")
        validos = grupo["estado"].eq("ok")

        # NaN mantiene visibles los huecos de ajustes no válidos
        r2 = grupo.reindex(columns=["R2"])["R2"].where(validos)
        rmse = grupo.reindex(columns=["RMSE_m"])["RMSE_m"].where(validos)

        axs[0].plot(grupo["theta"], r2, "o-", label=tramo)
        axs[1].plot(grupo["theta"], rmse, "o-", label=tramo)

    axs[0].set(xlabel="θ", ylabel="R²", title="Linealidad")
    axs[1].set(xlabel="θ", ylabel="RMSE (m)", title="Error del ajuste")

    for ax in axs:
        ax.axvline(
            THETA_AJUSTE,
            color="black",
            linestyle=":",
            label="θ de referencia",
        )
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

    fig.suptitle(f"{river_id} — {nombres[river_id]}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 3. Ajustes por segmentos en χ–elevación

Ajustamos los tramos definidos y calculamos su pendiente Mχ y ksn.
Los límites manuales no se consideran knickpoints detectados.

In [ ]:
# Usa THETA_AJUSTE definido en la celda 7.
# Si lo cambias, vuelve a ejecutar desde esa celda.

theta = round(THETA_AJUSTE, 8)

if any(theta not in datos for datos in chi_sensibilidad.values()):
    raise ValueError("Ejecuta nuevamente la celda 7 con este θ.")

filas_ajustes = []

for river_id, p in perfiles.items():
    chi = chi_sensibilidad[river_id][theta]
    z = p["elevacion_m"].to_numpy(dtype=float)
    x = p["distancia_m"].to_numpy(dtype=float)

    for tramo, mask in mascaras_tramos[river_id].items():
        ajuste = ajustar_recta(chi, z, x, mask)

        # Mχ es la pendiente z–χ; no es directamente ksn
        ksn_segmento = (
            ajuste["M_chi"] * A0_ANALISIS**theta
            if ajuste["estado"] == "ok" else np.nan
        )

        filas_ajustes.append({
            "ID": river_id,
            "rio": nombres[river_id],
            "tramo": tramo,
            "theta": theta,
            "A0_m2": A0_ANALISIS,
            "ksn_segmento": ksn_segmento,
            "delimitacion": (
                "manual" if river_id in TRAMOS else "completa_provisional"
            ),
            **ajuste,
        })

ajustes_segmentos = pd.DataFrame(filas_ajustes)
#display(ajustes_segmentos)


# ── MODIFICAR AQUÍ: solo visualización ───────────────────────────────────
RIOS_VER = ["6"]

# RIOS_VER = ["1", "3", "16"]
# RIOS_VER = list(perfiles)

for river_id in map(str, RIOS_VER):
    if river_id not in perfiles:
        raise ValueError(f"No está cargado el río {river_id}.")

    p = perfiles[river_id]
    chi = chi_sensibilidad[river_id][theta]
    z = p["elevacion_m"].to_numpy(dtype=float)

    fig, axs = plt.subplots(1, 2, figsize=(12, 5))
    axs[0].plot(chi, z, color="0.8", linewidth=1, label="Perfil completo")

    resultados = ajustes_segmentos[
        ajustes_segmentos["ID"].eq(river_id)
    ]

    for fila in resultados.to_dict("records"):
        if fila["estado"] != "ok":
            continue

        tramo = fila["tramo"]
        mask = mascaras_tramos[river_id][tramo]
        valid = mask & np.isfinite(z)

        prediccion = fila["intercepto_m"] + fila["M_chi"] * chi[valid]

        puntos = axs[0].plot(
            chi[valid], z[valid], ".", markersize=3, label=tramo
        )
        color = puntos[0].get_color()

        axs[0].plot(
            chi[valid], prediccion,
            color=color, linewidth=2,
        )
        axs[1].plot(
            chi[valid], z[valid] - prediccion,
            ".", color=color, markersize=3, label=tramo,
        )

    axs[0].set(
        xlabel="χ trapezoidal (m)",
        ylabel="Elevación (m)",
        title="Ajustes por tramo",
    )
    axs[1].set(
        xlabel="χ trapezoidal (m)",
        ylabel="Elevación observada − ajustada (m)",
        title="Residuos",
    )
    axs[1].axhline(0, color="black", linewidth=1)

    for ax in axs:
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

    fig.suptitle(
        f"{river_id} — {nombres[river_id]} | θ={theta}"
    )
    plt.tight_layout()
    plt.show()
    plt.close(fig)

### Comparación visual de θ

Comparamos el perfil χ–elevación y su ajuste lineal para tres concavidades.

In [ ]:
# ── MODIFICAR AQUÍ ──────────────────────────────────────────────────────
RIO_VER = "6"
THETAS_VER = [0.30, 0.45, 0.60]
TRAMO_VER = "completo_provisional"


# ── Comprobar datos ──────────────────────────────────────────────────────
RIO_VER = str(RIO_VER)

if RIO_VER not in perfiles:
    raise ValueError(f"No está cargado el río {RIO_VER}.")

if TRAMO_VER not in mascaras_tramos[RIO_VER]:
    disponibles = list(mascaras_tramos[RIO_VER])
    raise ValueError(f"Tramo no encontrado. Disponibles: {disponibles}")

p = perfiles[RIO_VER]
distancia = p["distancia_m"].to_numpy(dtype=float)
z = p["elevacion_m"].to_numpy(dtype=float)
mask = mascaras_tramos[RIO_VER][TRAMO_VER]

thetas_ver = list(dict.fromkeys(round(float(t), 8) for t in THETAS_VER))

if not thetas_ver:
    raise ValueError("Selecciona al menos un valor de θ.")

for theta in thetas_ver:
    if theta not in chi_sensibilidad[RIO_VER]:
        raise ValueError(
            f"θ={theta} no está calculado. Añádelo al barrido y ejecútalo."
        )


# ── Perfiles χ–elevación: sin gráficos de residuos ────────────────────────
fig, axes = plt.subplots(
    1,
    len(thetas_ver),
    figsize=(5 * len(thetas_ver), 5),
    sharey=True,
    squeeze=False,
)

for j, theta in enumerate(thetas_ver):
    ax = axes[0, j]
    chi = chi_sensibilidad[RIO_VER][theta]
    ajuste = ajustar_recta(chi, z, distancia, mask)

    valid = mask & np.isfinite(chi) & np.isfinite(z)

    ax.plot(
        chi,
        np.where(valid, z, np.nan),
        color="steelblue",
        linewidth=1.5,
        label="Perfil observado",
    )

    if ajuste["estado"] in ("ok", "pendiente_no_positiva"):
        prediccion = ajuste["intercepto_m"] + ajuste["M_chi"] * chi

        ax.plot(
            chi,
            np.where(valid, prediccion, np.nan),
            color="darkorange",
            linestyle="--",
            linewidth=1.8,
            label="Recta ajustada",
        )

        ax.set_title(
            f"θ = {theta:.2f}\n"
            f"R² = {ajuste['R2']:.3f} | RMSE = {ajuste['RMSE_m']:.1f} m"
        )
    else:
        ax.set_title(f"θ = {theta:.2f}\n{ajuste['estado']}")

    ax.set_xlabel("χ (m)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

axes[0, 0].set_ylabel("Elevación (m)")

fig.suptitle(
    f"{RIO_VER} — {nombres[RIO_VER]} | {TRAMO_VER}",
    fontsize=14,
)
fig.tight_layout(rect=(0, 0, 1, 0.94))
plt.show()
plt.close(fig)

## 4. Candidatos a knickpoints

Buscamos cambios de pendiente entre tramos vecinos en χ–elevación.
Los resultados son candidatos para revisión.

In [ ]:
from scipy.signal import find_peaks

# ── Parámetros iniciales: ajustables, no calibrados para tus ríos ─────────
THETA_DETECCION = round(float(THETA_AJUSTE), 8)

VENTANA_LADO_M = 500.0      # Longitud de cada tramo vecino
PASO_BUSQUEDA_M = 100.0     # Separación entre posiciones evaluadas
SEPARACION_CANDIDATOS_M = 500.0

MIN_PUNTOS_LADO = 10
R2_MINIMO = 0.80

# Cambio relativo = |ksn_arriba - ksn_abajo| / max(ksn_arriba, ksn_abajo)
CAMBIO_MINIMO = 0.30        # Diferencia de al menos 30 %
PROMINENCIA_MINIMA = 0.10   # Destacar frente a cambios vecinos

COLUMNAS_DETECCION = [
    "ID", "rio", "tramo", "bloque", "indice",
    "distancia_m", "x_utm", "y_utm", "elevacion_m", "chi_m",
    "theta", "ventana_lado_m",
    "ksn_abajo", "ksn_arriba", "cambio_relativo",
    "r2_abajo", "r2_arriba", "evaluacion_valida", "sentido",
]


def detectar_cambios(river_id, theta, ventana_m):
    """Detector local exploratorio; no modifica los perfiles originales."""
    theta = round(float(theta), 8)

    if theta not in chi_sensibilidad[river_id]:
        raise ValueError(f"Falta calcular θ={theta} para el río {river_id}.")

    p = perfiles[river_id]
    x = p["distancia_m"].to_numpy(dtype=float)
    z = p["elevacion_m"].to_numpy(dtype=float)
    chi = np.asarray(chi_sensibilidad[river_id][theta])

    evaluaciones = []
    candidatos = []

    for tramo, mask in mascaras_tramos[river_id].items():
        indices = np.flatnonzero(
            mask & np.isfinite(z) & np.isfinite(chi)
        )

        # No unir puntos a través de huecos sin datos
        bloques = np.split(
            indices, np.flatnonzero(np.diff(indices) > 1) + 1
        )

        for bloque_id, idx in enumerate(bloques):
            if len(idx) < 2 * MIN_PUNTOS_LADO:
                continue

            xb, zb, cb = x[idx], z[idx], chi[idx]

            if xb[-1] - xb[0] <= 2 * ventana_m:
                continue

            centros = np.arange(
                xb[0] + ventana_m,
                xb[-1] - ventana_m,
                PASO_BUSQUEDA_M,
            )

            filas_bloque = []

            for centro in centros:
                abajo = (xb >= centro - ventana_m) & (xb < centro)
                arriba = (xb > centro) & (xb <= centro + ventana_m)

                j = int(idx[np.argmin(np.abs(xb - centro))])

                fila = {
                    "ID": river_id,
                    "rio": nombres[river_id],
                    "tramo": tramo,
                    "bloque": bloque_id,
                    "indice": j,
                    "distancia_m": float(x[j]),
                    "x_utm": float(p.iloc[j]["x_utm"]),
                    "y_utm": float(p.iloc[j]["y_utm"]),
                    "elevacion_m": float(z[j]),
                    "chi_m": float(chi[j]),
                    "theta": theta,
                    "ventana_lado_m": ventana_m,
                    "ksn_abajo": np.nan,
                    "ksn_arriba": np.nan,
                    "cambio_relativo": np.nan,
                    "r2_abajo": np.nan,
                    "r2_arriba": np.nan,
                    "evaluacion_valida": False,
                    "sentido": "",
                }

                suficientes = (
                    abajo.sum() >= MIN_PUNTOS_LADO
                    and arriba.sum() >= MIN_PUNTOS_LADO
                )

                if suficientes:
                    fit_a = linregress(cb[abajo], zb[abajo])
                    fit_b = linregress(cb[arriba], zb[arriba])

                    ka = float(fit_a.slope * A0_ANALISIS**theta)
                    kb = float(fit_b.slope * A0_ANALISIS**theta)
                    r2a = float(fit_a.rvalue**2)
                    r2b = float(fit_b.rvalue**2)

                    valido = (
                        np.isfinite([ka, kb, r2a, r2b]).all()
                        and ka > 0 and kb > 0
                        and min(r2a, r2b) >= R2_MINIMO
                    )

                    fila.update({
                        "ksn_abajo": ka,
                        "ksn_arriba": kb,
                        "r2_abajo": r2a,
                        "r2_arriba": r2b,
                        "evaluacion_valida": valido,
                    })

                    if valido:
                        fila["cambio_relativo"] = abs(kb - ka) / max(ka, kb)
                        fila["sentido"] = (
                            "aumenta_aguas_arriba"
                            if kb > ka else "disminuye_aguas_arriba"
                        )

                filas_bloque.append(fila)

            if not filas_bloque:
                continue

            tabla = pd.DataFrame(filas_bloque)
            señal = np.where(
                tabla["evaluacion_valida"],
                tabla["cambio_relativo"],
                0.0,
            )

            picos, _ = find_peaks(
                señal,
                height=CAMBIO_MINIMO,
                prominence=PROMINENCIA_MINIMA,
                distance=max(
                    1,
                    int(np.ceil(
                        SEPARACION_CANDIDATOS_M / PASO_BUSQUEDA_M
                    )),
                ),
            )

            for k in picos:
                # Evitar candidatos inmediatamente junto a ajustes inválidos
                if tabla.iloc[k-1:k+2]["evaluacion_valida"].all():
                    candidatos.append(filas_bloque[k].copy())

            evaluaciones.extend(filas_bloque)

    return (
        pd.DataFrame(evaluaciones, columns=COLUMNAS_DETECCION),
        pd.DataFrame(candidatos, columns=COLUMNAS_DETECCION),
    )


# ── Ejecutar para todos los perfiles cargados ────────────────────────────
lista_evaluaciones = []
lista_candidatos = []

for river_id in perfiles:
    evaluacion, candidatos = detectar_cambios(
        river_id, THETA_DETECCION, VENTANA_LADO_M
    )
    lista_evaluaciones.append(evaluacion)
    lista_candidatos.append(candidatos)

    print(f"{river_id} — {nombres[river_id]}: {len(candidatos)} candidatos")

evaluaciones_base = pd.concat(lista_evaluaciones, ignore_index=True)
candidatos_base = pd.concat(lista_candidatos, ignore_index=True)

# Identificadores válidos dentro de esta ejecución
candidatos_base["candidato"] = [
    f"R{int(r.ID):02d}_C{j+1:03d}"
    for j, r in candidatos_base.iterrows()
]

display(candidatos_base[
  [
        "candidato", "rio", "tramo", "distancia_m", "elevacion_m",
        "ksn_abajo", "ksn_arriba", "cambio_relativo", "sentido",
    ]
])

### Comparación de métodos

- **Naranja:** cambios locales de pendiente.
- **Morado:** TopoToolbox — knickpointfinder.
- **Verde:** Gailleton — resultados de LSDTopoTools.

La franja de 180–220 m es una referencia visual.
Los puntos son candidatos pendientes de revisión.

In [ ]:
import numpy as np
import pandas as pd
import topotoolbox as tt

# ── MODIFICAR AQUÍ ────────────────────────────────────────────────
TOLERANCIA_TT_M = 20.0  # Tolerancia vertical, no cota del knickpoint

# Mayor tolerancia → normalmente menos candidatos.
# Valor inicial exploratorio: revisar según la calidad del DEM.


class Perfil1DTopoToolbox(tt.StreamObject):
    """
    Adapta un perfil ya extraído al algoritmo oficial de TopoToolbox.
    Uso limitado a knickpointfinder, imposemin y lowerenv.
    """

    def __init__(self, distancia):
        d = np.asarray(distancia, dtype=float)

        if (
            len(d) < 3
            or not np.isfinite(d).all()
            or np.any(np.diff(d) <= 0)
        ):
            raise ValueError("Se requieren distancias finitas y crecientes.")

        # Orden interno: cabecera → outlet.
        self.stream = np.arange(len(d), dtype=np.int64)
        self.source = np.arange(len(d) - 1, dtype=np.int64)
        self.target = self.source + 1
        self._distancia = np.ascontiguousarray(
            (d - d[0])[::-1], dtype=np.float32
        )

        if np.any(np.diff(self._distancia) >= 0):
            raise ValueError("Las distancias pierden precisión en float32.")

    def ezgetnal(self, valores, dtype=None):
        a = np.array(valores, dtype=dtype, copy=True)

        if a.shape != self.stream.shape:
            raise ValueError("Elevaciones y nodos no tienen igual longitud.")

        return a

    def upstream_distance(self):
        return self._distancia.copy()

    def node_to_node_distance(self):
        return (
            self._distancia[self.source]
            - self._distancia[self.target]
        )


if not np.isfinite(TOLERANCIA_TT_M) or TOLERANCIA_TT_M <= 0:
    raise ValueError("TOLERANCIA_TT_M debe ser positiva y finita.")

filas_tt = []
filas_qc_tt = []

# Calcula TODOS los ríos, independientemente de la selección gráfica.
for river_id, p in perfiles.items():
    x = p["distancia_m"].to_numpy(float)
    z = p["elevacion_m"].to_numpy(float)
    chi = np.asarray(chi_sensibilidad[river_id][THETA_DETECCION])

    for tramo, mascara in mascaras_tramos[river_id].items():
        indices = np.flatnonzero(
            mascara & np.isfinite(z) & np.isfinite(chi)
        )

        # No unir sectores separados por datos faltantes.
        bloques = np.split(
            indices, np.flatnonzero(np.diff(indices) > 1) + 1
        )

        for bloque, idx in enumerate(bloques):
            if len(idx) < 3:
                continue

            cauce = Perfil1DTopoToolbox(x[idx])
            entrada = np.ascontiguousarray(
                z[idx][::-1], dtype=np.float32
            )

            z_tt = tt.imposemin(cauce, entrada)[::-1]

            detectados = cauce.knickpointfinder(
                entrada,
                knickpoints=np.zeros(len(idx), dtype=bool),
                tolerance=TOLERANCIA_TT_M,
            )[::-1]

            for k, j in enumerate(idx):
                filas_qc_tt.append({
                    "ID": river_id,
                    "tramo": tramo,
                    "indice": int(j),
                    "elevacion_original_m": z[j],
                    "elevacion_tt_m": float(z_tt[k]),
                    "correccion_tt_m": float(z_tt[k] - z[j]),
                })

                if not detectados[k]:
                    continue

                filas_tt.append({
                    "ID": river_id,
                    "rio": nombres[river_id],
                    "tramo": tramo,
                    "bloque": bloque,
                    "indice": int(j),
                    "distancia_m": x[j],
                    "elevacion_m": z[j],
                    "chi_m": chi[j],
                    "x_utm": float(p.iloc[j]["x_utm"]),
                    "y_utm": float(p.iloc[j]["y_utm"]),
                    "tolerancia_tt_m": TOLERANCIA_TT_M,
                    "correccion_tt_m": float(z_tt[k] - z[j]),
                })

columnas_tt = [
    "ID", "rio", "tramo", "bloque", "indice",
    "distancia_m", "elevacion_m", "chi_m", "x_utm", "y_utm",
    "tolerancia_tt_m", "correccion_tt_m",
]

candidatos_tt = pd.DataFrame(filas_tt, columns=columnas_tt)
qc_topotoolbox = pd.DataFrame(filas_qc_tt)

print(f"TopoToolbox: {len(candidatos_tt)} candidatos.")
print("Las elevaciones originales se conservan en los gráficos.")

# Descomenta para revisar las correcciones internas:
display(qc_topotoolbox)

In [ ]:
from pathlib import Path
from scipy.spatial import cKDTree

# ── MODIFICAR CUANDO EXISTAN RESULTADOS DE LSDTopoTools ─────────────
# Los archivos deben estar en el mismo CRS que tus perfiles: EPSG:32718.
# Usa la misma concavidad que THETA_DETECCION.
#
# Ejemplo de formato; los identificadores son solo ilustrativos:
#
# RESULTADOS_GAILLETON = {
#     "6": {
#         "archivo": r"C:\ruta\Reloca_ksnkp.csv",
#         "source_key": 0,
#         "basin_key": 0,
#         "theta": 0.45,
#         "crs": "EPSG:32718",
#     }
# }

RESULTADOS_GAILLETON = {}

# Distancia máxima entre un candidato de LSDTopoTools y tu perfil.
# Revisar según resolución del DEM y diferencias entre redes.
DISTANCIA_MAX_G_M = 30.0

filas_g = []
rechazos_g = []
estado_gailleton = {}

for river_id, p in perfiles.items():
    if river_id not in RESULTADOS_GAILLETON:
        estado_gailleton[river_id] = "pendiente"
        continue

    cfg = RESULTADOS_GAILLETON[river_id]

    if cfg["crs"] != "EPSG:32718":
        raise ValueError("Transforma y verifica las coordenadas antes de importar.")

    if not np.isclose(cfg["theta"], THETA_DETECCION):
        raise ValueError(f"Río {river_id}: las concavidades no coinciden.")

    archivo = Path(cfg["archivo"])

    if not archivo.name.endswith("_ksnkp.csv"):
        raise ValueError("Selecciona el archivo final terminado en _ksnkp.csv.")

    datos = pd.read_csv(archivo)
    requeridas = {
        "X", "Y", "elevation", "source_key", "basin_key",
        "delta_ksn", "delta_segelev",
    }

    if requeridas - set(datos.columns):
        raise ValueError(f"Faltan columnas: {requeridas - set(datos.columns)}")

    # Verificar que el cauce fue procesado, incluso si no tuvo candidatos.
    archivo_segmentos = archivo.with_name(
        archivo.name.replace("_ksnkp.csv", "_ksnkp_mchi.csv")
    )
    segmentos = pd.read_csv(archivo_segmentos)

    canal = (
        segmentos["source_key"].eq(cfg["source_key"])
        & segmentos["basin_key"].eq(cfg["basin_key"])
    )

    if not canal.any():
        raise ValueError(f"Río {river_id}: ese cauce no aparece en los resultados.")

    datos = datos[
        datos["source_key"].eq(cfg["source_key"])
        & datos["basin_key"].eq(cfg["basin_key"])
    ]

    xy = p[["x_utm", "y_utm"]].to_numpy(float)

    if not np.isfinite(xy).all():
        raise ValueError(f"Río {river_id}: coordenadas inválidas.")

    arbol = cKDTree(xy)

    for fila, r in datos.iterrows():
        registro = {
            "ID": river_id,
            "archivo": str(archivo),
            "fila_lsd": int(fila),
            "x_lsd": r["X"],
            "y_lsd": r["Y"],
            "elevacion_lsd_m": r["elevation"],
            "delta_ksn_lsd": r["delta_ksn"],
            "delta_segelev_lsd": r["delta_segelev"],
        }

        if not np.isfinite([r["X"], r["Y"]]).all():
            rechazos_g.append({
                **registro, "motivo": "coordenadas_invalidas"
            })
            continue

        distancia, j = arbol.query([r["X"], r["Y"]])
        j = int(j)

        tramos = [
            t for t, mascara in mascaras_tramos[river_id].items()
            if mascara[j]
        ]

        if (
            distancia > DISTANCIA_MAX_G_M
            or len(tramos) != 1
            or not np.isfinite(p.iloc[j]["elevacion_m"])
        ):
            rechazos_g.append({
                **registro,
                "desajuste_xy_m": distancia,
                "motivo": "sin_asociacion_valida_al_perfil",
            })
            continue

        punto = p.iloc[j]

        filas_g.append({
            **registro,
            "rio": nombres[river_id],
            "tramo": tramos[0],
            "indice": j,
            "distancia_m": punto["distancia_m"],
            "elevacion_m": punto["elevacion_m"],
            "chi_m": chi_sensibilidad[river_id][THETA_DETECCION][j],
            "x_utm": punto["x_utm"],
            "y_utm": punto["y_utm"],
            "desajuste_xy_m": distancia,
        })

    estado_gailleton[river_id] = "importado"

columnas_g = [
    "ID", "rio", "tramo", "indice", "distancia_m", "elevacion_m",
    "chi_m", "x_utm", "y_utm", "archivo", "fila_lsd",
    "x_lsd", "y_lsd", "elevacion_lsd_m",
    "delta_ksn_lsd", "delta_segelev_lsd", "desajuste_xy_m",
]

candidatos_gailleton = pd.DataFrame(filas_g, columns=columnas_g)
rechazos_gailleton = pd.DataFrame(rechazos_g)

print(f"Gailleton: {len(candidatos_gailleton)} candidatos importados.")
print(
    "Ríos pendientes:",
    sum(estado == "pendiente" for estado in estado_gailleton.values()),
)

## 5. Sensibilidad y ubicación espacial

Repetimos la detección con distintas concavidades y ventanas.
Comparamos posiciones y revisamos los candidatos sobre el cauce.

In [ ]:
# ── MODIFICAR AQUÍ ──────────────────────────────────────────────────────
THETAS_COMPARAR = [0.30, 0.45, 0.60]
VENTANAS_LADO_COMPARAR_M = [250.0, 500.0, 1000.0]

# Distancia máxima para asociar detecciones de distintas configuraciones
TOLERANCIA_ASOCIACION_M = 300.0

# Incluir también la configuración original
configuraciones = sorted(set(
    [
        (round(float(t), 8), float(w))
        for t in THETAS_COMPARAR
        for w in VENTANAS_LADO_COMPARAR_M
    ]
    + [(THETA_DETECCION, VENTANA_LADO_M)]
))

if any(w <= 0 for _, w in configuraciones):
    raise ValueError("Las ventanas deben ser positivas.")

for river_id in perfiles:
    for theta, _ in configuraciones:
        if theta not in chi_sensibilidad[river_id]:
            raise ValueError(
                f"Falta θ={theta} para {river_id}. "
                "Añádelo al barrido del punto 2."
            )

listas_pruebas = []
listas_evaluaciones = []

for theta, ventana in configuraciones:
    for river_id in perfiles:
        evaluacion, candidatos = detectar_cambios(
            river_id, theta, ventana
        )
        listas_evaluaciones.append(evaluacion)
        listas_pruebas.append(candidatos)

    print(f"Completado: θ={theta:.2f}, ventana por lado={ventana:g} m")

pruebas_candidatos = pd.concat(listas_pruebas, ignore_index=True)
evaluaciones_sensibilidad = pd.concat(
    listas_evaluaciones, ignore_index=True
)

# ── Asociar cada detección como máximo a un candidato de referencia ──────
coincidencias = {
    nombre: [] for nombre in candidatos_base["candidato"]
}
configuraciones_evaluables = {
    nombre: 0 for nombre in candidatos_base["candidato"]
}

for theta, ventana in configuraciones:
    detecciones = pruebas_candidatos[
        pruebas_candidatos["theta"].eq(theta)
        & pruebas_candidatos["ventana_lado_m"].eq(ventana)
    ]

    evaluaciones = evaluaciones_sensibilidad[
        evaluaciones_sensibilidad["theta"].eq(theta)
        & evaluaciones_sensibilidad["ventana_lado_m"].eq(ventana)
    ]

    pares = []

    for _, candidato in candidatos_base.iterrows():
        cerca = evaluaciones[
            evaluaciones["ID"].eq(candidato["ID"])
            & evaluaciones["tramo"].eq(candidato["tramo"])
            & evaluaciones["bloque"].eq(candidato["bloque"])
            & (
                evaluaciones["distancia_m"] - candidato["distancia_m"]
            ).abs().le(TOLERANCIA_ASOCIACION_M)
        ]

        # Cuenta posiciones evaluadas, aunque el ajuste no superara filtros
        if not cerca.empty:
            configuraciones_evaluables[candidato["candidato"]] += 1

        compatibles = detecciones[
            detecciones["ID"].eq(candidato["ID"])
            & detecciones["tramo"].eq(candidato["tramo"])
            & detecciones["bloque"].eq(candidato["bloque"])
            & detecciones["sentido"].eq(candidato["sentido"])
        ]

        for idx, deteccion in compatibles.iterrows():
            diferencia = abs(
                deteccion["distancia_m"] - candidato["distancia_m"]
            )
            if diferencia <= TOLERANCIA_ASOCIACION_M:
                pares.append((
                    diferencia,
                    candidato["candidato"],
                    idx,
                    float(deteccion["distancia_m"]),
                ))

    usados_base, usados_prueba = set(), set()

    for diferencia, nombre, idx, posicion in sorted(pares):
        if nombre not in usados_base and idx not in usados_prueba:
            coincidencias[nombre].append(posicion)
            usados_base.add(nombre)
            usados_prueba.add(idx)

catalogo = candidatos_base.copy()

catalogo["configuraciones_probadas"] = len(configuraciones)
catalogo["configuraciones_evaluables"] = [
    configuraciones_evaluables[c] for c in catalogo["candidato"]
]
catalogo["detecciones_asociadas"] = [
    len(coincidencias[c]) for c in catalogo["candidato"]
]
catalogo["posicion_min_m"] = [
    min(coincidencias[c]) if coincidencias[c] else np.nan
    for c in catalogo["candidato"]
]
catalogo["posicion_max_m"] = [
    max(coincidencias[c]) if coincidencias[c] else np.nan
    for c in catalogo["candidato"]
]

catalogo["revision"] = "pendiente"
catalogo["contexto_geologico"] = "no_revisado"
catalogo["observaciones"] = ""

display(catalogo[
    [
        "candidato", "rio", "distancia_m",
        "detecciones_asociadas", "configuraciones_probadas",
        "configuraciones_evaluables", "posicion_min_m", "posicion_max_m",
    ]
])

### Visualizar perfiles

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ── MODIFICAR AQUÍ: SOLO VISUALIZACIÓN ─────────────────────────────
#RIOS_VER = ["6"]
# RIOS_VER = ["1", "3", "16"]
RIOS_VER = list(perfiles)

METODOS_VER = ["Local", "TopoToolbox", "Gailleton"]

MOSTRAR_ETIQUETAS = True
MOSTRAR_FRANJA = True
ELEVACION_MIN = 180
ELEVACION_MAX = 220

ESTILOS = {
    "Local":      {"color": "#D55E00", "marker": "o", "s": 65},
    "TopoToolbox": {"color": "#7B2CBF", "marker": "^", "s": 100},
    "Gailleton":  {"color": "#009E73", "marker": "s", "s": 135},
}

tablas = []

for metodo, tabla, prefijo in [
    ("Local", candidatos_base, "L"),
    ("TopoToolbox", candidatos_tt, "T"),
    ("Gailleton", candidatos_gailleton, "G"),
]:
    copia = tabla.copy()
    copia["ID"] = copia["ID"].astype(str)
    copia["metodo"] = metodo

    # Numeración independiente para cada río y método.
    numero = copia.groupby("ID").cumcount() + 1
    copia["etiqueta"] = [
        f"{prefijo}{n:03d}" for n in numero
    ]
    tablas.append(copia)

catalogo_metodos = pd.concat(tablas, ignore_index=True)


def figura_revision(river_id):
    river_id = str(river_id)

    if river_id not in perfiles:
        raise ValueError(f"No está cargado el río {river_id}.")

    if set(METODOS_VER) - set(ESTILOS):
        raise ValueError("Revisa los nombres en METODOS_VER.")

    p = perfiles[river_id]
    chi = chi_sensibilidad[river_id][THETA_DETECCION]
    candidatos = catalogo_metodos[
        catalogo_metodos["ID"].eq(river_id)
    ]

    fig, axs = plt.subplots(1, 3, figsize=(18, 5.7))

    axs[0].plot(
        p["distancia_m"] / 1000, p["elevacion_m"],
        color="steelblue", linewidth=1.2,
    )
    axs[1].plot(
        chi, p["elevacion_m"],
        color="steelblue", linewidth=1.2,
    )
    axs[2].plot(
        p["x_utm"], p["y_utm"],
        color="steelblue", linewidth=1.2,
    )
    axs[2].scatter(
        p.iloc[0]["x_utm"], p.iloc[0]["y_utm"],
        marker="*", s=100, color="black", zorder=6,
    )

    leyenda = []

    for orden, metodo in enumerate(METODOS_VER):
        estilo = ESTILOS[metodo]
        grupo = candidatos[candidatos["metodo"].eq(metodo)]

        pendiente = (
            metodo == "Gailleton"
            and estado_gailleton[river_id] == "pendiente"
        )
        cantidad = "pendiente" if pendiente else str(len(grupo))

        leyenda.append(Line2D(
            [], [],
            linestyle="none",
            marker=estilo["marker"],
            color=estilo["color"],
            markerfacecolor="none",
            markersize=8,
            label=f"{metodo}: {cantidad}",
        ))

        for _, candidato in grupo.iterrows():
            puntos = [
                (
                    candidato["distancia_m"] / 1000,
                    candidato["elevacion_m"],
                ),
                (candidato["chi_m"], candidato["elevacion_m"]),
                (candidato["x_utm"], candidato["y_utm"]),
            ]

            for ax, (x, y) in zip(axs, puntos):
                # Símbolos vacíos para distinguir métodos coincidentes.
                ax.scatter(
                    x, y,
                    marker=estilo["marker"],
                    s=estilo["s"],
                    facecolors="none",
                    edgecolors=estilo["color"],
                    linewidths=1.6,
                    zorder=7 + orden,
                )

                if MOSTRAR_ETIQUETAS:
                    ax.annotate(
                        candidato["etiqueta"],
                        (x, y),
                        xytext=(5, 8 + 10 * orden),
                        textcoords="offset points",
                        fontsize=7,
                        color=estilo["color"],
                    )

    if MOSTRAR_FRANJA:
        # Solo perfiles: el mapa tiene Norte, no elevación, en el eje Y.
        for ax in axs[:2]:
            ax.axhspan(
                ELEVACION_MIN, ELEVACION_MAX,
                color="gold", alpha=0.22, zorder=0,
            )
            for elevacion in (ELEVACION_MIN, ELEVACION_MAX):
                ax.axhline(
                    elevacion, color="darkorange",
                    linestyle="--", linewidth=0.8,
                )

    axs[0].set(
        title="Perfil longitudinal",
        xlabel="Distancia desde el outlet (km)",
        ylabel="Elevación (m)",
    )
    axs[1].set(
        title=f"χ–elevación | θ={THETA_DETECCION}",
        xlabel="χ (m)",
        ylabel="Elevación (m)",
    )
    axs[2].set(
        title="Ubicación sobre el cauce",
        xlabel="Este (m)",
        ylabel="Norte (m)",
    )
    axs[2].set_aspect("equal")
    axs[2].ticklabel_format(style="plain", useOffset=False)

    for ax in axs:
        ax.grid(alpha=0.25)

    fig.suptitle(
        f"{river_id} — {nombres[river_id]} | "
        "Candidatos por método"
    )
    fig.legend(
        handles=leyenda,
        loc="lower center",
        ncol=len(METODOS_VER),
        frameon=False,
    )
    fig.tight_layout(rect=(0, 0.07, 1, 0.95))

    return fig


for river_id in map(str, RIOS_VER):
    fig = figura_revision(river_id)
    plt.show()
    plt.close(fig)

## 6. Sensibilidad de TopoToolbox

Comparamos tolerancias verticales y registramos qué candidatos
se detectan cerca de la misma posición.

In [ ]:
# ── MODIFICAR AQUÍ ────────────────────────────────────────────────
TOLERANCIAS_TT_M = [10.0, 20.0, 40.0]

# Distancia sobre el cauce para asociar candidatos entre ejecuciones.
# Es un parámetro exploratorio: revisar según resolución y morfología.
DISTANCIA_ASOCIACION_TT_M = 300.0


def ejecutar_tt_tolerancia(tolerancia):
    """Ejecuta el método oficial sobre los mismos perfiles y tramos."""
    filas = []

    for river_id, p in perfiles.items():
        x = p["distancia_m"].to_numpy(float)
        z = p["elevacion_m"].to_numpy(float)
        chi = np.asarray(
            chi_sensibilidad[river_id][THETA_DETECCION]
        )

        for tramo, mascara in mascaras_tramos[river_id].items():
            indices = np.flatnonzero(
                mascara & np.isfinite(z) & np.isfinite(chi)
            )
            bloques = np.split(
                indices,
                np.flatnonzero(np.diff(indices) > 1) + 1,
            )

            for bloque, idx in enumerate(bloques):
                if len(idx) < 3:
                    continue

                cauce = Perfil1DTopoToolbox(x[idx])
                entrada = np.ascontiguousarray(
                    z[idx][::-1], dtype=np.float32
                )

                detectados = cauce.knickpointfinder(
                    entrada,
                    knickpoints=np.zeros(len(idx), dtype=bool),
                    tolerance=float(tolerancia),
                )[::-1]

                for j in idx[detectados]:
                    filas.append({
                        "ID": str(river_id),
                        "tramo": tramo,
                        "bloque": bloque,
                        "indice": int(j),
                        "distancia_m": float(x[j]),
                        "tolerancia_tt_m": float(tolerancia),
                    })

    return pd.DataFrame(filas, columns=[
        "ID", "tramo", "bloque", "indice",
        "distancia_m", "tolerancia_tt_m",
    ])


tolerancias = sorted(set(
    map(float, TOLERANCIAS_TT_M + [TOLERANCIA_TT_M])
))

if any(not np.isfinite(t) or t <= 0 for t in tolerancias):
    raise ValueError("Las tolerancias deben ser positivas y finitas.")

if (
    not np.isfinite(DISTANCIA_ASOCIACION_TT_M)
    or DISTANCIA_ASOCIACION_TT_M < 0
):
    raise ValueError("Revisa DISTANCIA_ASOCIACION_TT_M.")

resultados_tt = []

for tolerancia in tolerancias:
    # Reutilizar la ejecución de referencia.
    if np.isclose(tolerancia, TOLERANCIA_TT_M):
        tabla = candidatos_tt[
            ["ID", "tramo", "bloque", "indice", "distancia_m"]
        ].copy()
        tabla["ID"] = tabla["ID"].astype(str)
        tabla["tolerancia_tt_m"] = tolerancia
    else:
        tabla = ejecutar_tt_tolerancia(tolerancia)

    resultados_tt.append(tabla)
    print(f"Tolerancia {tolerancia:g} m: {len(tabla)} candidatos.")

pruebas_tt = pd.concat(resultados_tt, ignore_index=True)

# Asociaciones uno a uno por configuración, priorizando menor distancia.
referencia_tt = candidatos_tt.reset_index(drop=True).copy()
referencia_tt["ID"] = referencia_tt["ID"].astype(str)

asociaciones = {i: [] for i in referencia_tt.index}

for tolerancia in tolerancias:
    prueba = pruebas_tt[
        pruebas_tt["tolerancia_tt_m"].eq(tolerancia)
    ]

    pares = []

    for i, base in referencia_tt.iterrows():
        compatibles = prueba[
            prueba["ID"].eq(base["ID"])
            & prueba["tramo"].eq(base["tramo"])
            & prueba["bloque"].eq(base["bloque"])
        ]

        for j, candidato in compatibles.iterrows():
            diferencia = abs(
                candidato["distancia_m"] - base["distancia_m"]
            )

            if diferencia <= DISTANCIA_ASOCIACION_TT_M:
                pares.append((
                    diferencia, i, j, candidato["distancia_m"]
                ))

    usados_base, usados_prueba = set(), set()

    for diferencia, i, j, posicion in sorted(pares):
        if i in usados_base or j in usados_prueba:
            continue

        asociaciones[i].append((tolerancia, posicion))
        usados_base.add(i)
        usados_prueba.add(j)

sensibilidad_tt = referencia_tt.copy()

sensibilidad_tt["n_tolerancias"] = len(tolerancias)
sensibilidad_tt["n_detecciones"] = [
    len(asociaciones[i]) for i in referencia_tt.index
]
sensibilidad_tt["tolerancias_detectado"] = [
    ", ".join(f"{t:g}" for t, _ in asociaciones[i])
    for i in referencia_tt.index
]
sensibilidad_tt["posicion_min_m"] = [
    min(p for _, p in asociaciones[i])
    if asociaciones[i] else np.nan
    for i in referencia_tt.index
]
sensibilidad_tt["posicion_max_m"] = [
    max(p for _, p in asociaciones[i])
    if asociaciones[i] else np.nan
    for i in referencia_tt.index
]

display(sensibilidad_tt[[
    "ID", "tramo", "indice", "distancia_m",
    "n_detecciones", "n_tolerancias",
    "tolerancias_detectado", "posicion_min_m", "posicion_max_m",
]])

# La frecuencia de detección NO es una probabilidad de que sea real.

In [ ]:
import hashlib

catalogo_revision = catalogo_metodos.copy().reset_index(drop=True)
catalogo_revision["ID"] = catalogo_revision["ID"].astype(str)

prefijos = {
    "Local": "L",
    "TopoToolbox": "T",
    "Gailleton": "G",
}


def crear_id_kp(fila):
    # El índice corresponde a la fila del perfil original.
    base = (
        f"R{int(fila['ID']):02d}_"
        f"{prefijos[fila['metodo']]}_"
        f"I{int(fila['indice']):05d}"
    )

    # Gailleton puede asociar varios registros al mismo nodo del perfil.
    if fila["metodo"] == "Gailleton":
        origen = (
            f"{fila.get('archivo', '')}|"
            f"{fila.get('fila_lsd', '')}"
        )
        sufijo = hashlib.sha1(
            origen.encode("utf-8")
        ).hexdigest()[:6]
        base += f"_{sufijo}"

    return base


catalogo_revision["id_kp"] = [
    crear_id_kp(fila)
    for _, fila in catalogo_revision.iterrows()
]

if catalogo_revision["id_kp"].duplicated().any():
    raise ValueError(
        "Hay IDs duplicados. Revisa si existen tramos superpuestos "
        "o detecciones duplicadas antes de continuar."
    )

# ── TU SELECCIÓN PERSONAL ─────────────────────────────────────────
IDS_ME_GUSTAN = []

# Quita # y reemplaza estos ejemplos por IDs reales de tu catálogo:
# IDS_ME_GUSTAN = ["R06_T_I00452", "R01_L_I00730"]

catalogo_revision["me_interesa"] = (
    catalogo_revision["id_kp"].isin(IDS_ME_GUSTAN)
)

desconocidos = set(IDS_ME_GUSTAN) - set(catalogo_revision["id_kp"])

if desconocidos:
    print("IDs que no aparecen en esta ejecución:", sorted(desconocidos))

# Anotaciones opcionales. Usa IDs reales.
REVISION_MANUAL = {
    # "R06_T_I00452": {
    #     "revision": "pendiente",
    #     "observaciones": "Revisar contacto litológico en QGIS",
    # },
}

catalogo_revision["revision"] = "pendiente"
catalogo_revision["observaciones"] = ""

for id_kp, datos in REVISION_MANUAL.items():
    mascara = catalogo_revision["id_kp"].eq(id_kp)

    if not mascara.any():
        print(f"Anotación sin candidato correspondiente: {id_kp}")
        continue

    for columna in ["revision", "observaciones"]:
        if columna in datos:
            catalogo_revision.loc[mascara, columna] = datos[columna]

# Incorporar la sensibilidad de TopoToolbox.
columnas_sensibilidad = [
    "ID", "tramo", "indice",
    "n_detecciones", "n_tolerancias", "tolerancias_detectado",
    "posicion_min_m", "posicion_max_m",
]

sens = sensibilidad_tt[columnas_sensibilidad].copy()
sens["metodo"] = "TopoToolbox"

catalogo_revision = catalogo_revision.merge(
    sens,
    on=["ID", "tramo", "indice", "metodo"],
    how="left",
    validate="many_to_one",
)

# Actualiza las etiquetas de la función figura_revision que ya tienes.
catalogo_metodos = catalogo_revision.copy()
catalogo_metodos["etiqueta"] = catalogo_metodos["id_kp"]

# ── MODIFICAR AQUÍ: tabla que quieres ver ──────────────────────────
#RIOS_TABLA = ["6"]
RIOS_TABLA = list(perfiles)

columnas_mostrar = [
    "id_kp", "rio", "metodo", "distancia_m", "elevacion_m",
    "me_interesa", "revision", "n_detecciones", "n_tolerancias",
]

display(
    catalogo_revision.loc[
        catalogo_revision["ID"].isin(map(str, RIOS_TABLA)),
        columnas_mostrar,
    ].sort_values(["rio", "distancia_m", "metodo"])
)

## 8. Coincidencias entre métodos

Comparamos candidatos cercanos sobre el mismo tramo.
La proximidad indica una posible correspondencia, no una confirmación.

In [ ]:
from itertools import combinations

# ── MODIFICAR AQUÍ ────────────────────────────────────────────────
DISTANCIA_COINCIDENCIA_M = 300.0

if (
    not np.isfinite(DISTANCIA_COINCIDENCIA_M)
    or DISTANCIA_COINCIDENCIA_M < 0
):
    raise ValueError("La distancia debe ser finita y no negativa.")

filas_coincidencias = []

for (river_id, tramo), grupo in catalogo_revision.groupby(
    ["ID", "tramo"], sort=False
):
    metodos = sorted(grupo["metodo"].unique())

    for metodo_a, metodo_b in combinations(metodos, 2):
        a = grupo[grupo["metodo"].eq(metodo_a)]
        b = grupo[grupo["metodo"].eq(metodo_b)]

        posibles = []

        for _, ca in a.iterrows():
            for _, cb in b.iterrows():
                separacion = abs(
                    ca["distancia_m"] - cb["distancia_m"]
                )

                if separacion <= DISTANCIA_COINCIDENCIA_M:
                    posibles.append((
                        separacion, ca["id_kp"], cb["id_kp"]
                    ))

        # Un candidato se asocia como máximo a uno del otro método.
        usados_a, usados_b = set(), set()

        for separacion, id_a, id_b in sorted(posibles):
            if id_a in usados_a or id_b in usados_b:
                continue

            filas_coincidencias.append({
                "ID": river_id,
                "tramo": tramo,
                "metodo_a": metodo_a,
                "id_a": id_a,
                "metodo_b": metodo_b,
                "id_b": id_b,
                "separacion_m": separacion,
            })

            usados_a.add(id_a)
            usados_b.add(id_b)

coincidencias_metodos = pd.DataFrame(
    filas_coincidencias,
    columns=[
        "ID", "tramo", "metodo_a", "id_a",
        "metodo_b", "id_b", "separacion_m",
    ],
)

# Guardar las asociaciones directas sin fusionar candidatos.
vecinos = {
    id_kp: [] for id_kp in catalogo_revision["id_kp"]
}

for _, fila in coincidencias_metodos.iterrows():
    vecinos[fila["id_a"]].append(fila["id_b"])
    vecinos[fila["id_b"]].append(fila["id_a"])

catalogo_revision["coincide_con"] = [
    "; ".join(vecinos[id_kp])
    for id_kp in catalogo_revision["id_kp"]
]

display(coincidencias_metodos)

# Estas asociaciones no agrupan automáticamente tres puntos
# como un único knickpoint ni indican un origen geológico.

### Selección manual

In [ ]:
# Selección manual ANTES de la sección 9.
# Abre una ventana independiente; no utiliza ipympl ni widgets.
%matplotlib tk

import matplotlib
print("Modo gráfico:", matplotlib.get_backend())
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── MODIFICAR AQUÍ ────────────────────────────────────────────────
RIOS_MANUALES = list(perfiles)
#RIOS_MANUALES = ["1"]

ARCHIVO_CACHE_MANUAL = (
    BASE / "analisis_knickpoints"
    / "cache" / "knickpoints_manuales.json"
)

ARCHIVO_CACHE_MANUAL.parent.mkdir(parents=True, exist_ok=True)

# Identificar la versión de cada perfil.
huellas = {}

for rid, p in perfiles.items():
    datos = np.asarray(
        p[[
            "distancia_m", "elevacion_m", "area_m2",
            "x_utm", "y_utm",
        ]].to_numpy(float),
        dtype="<f8",
        order="C",
    )
    huellas[str(rid)] = hashlib.sha256(datos.tobytes()).hexdigest()


def leer_cache_manual():
    if not ARCHIVO_CACHE_MANUAL.exists():
        return []

    contenido = json.loads(
        ARCHIVO_CACHE_MANUAL.read_text(encoding="utf-8")
    )

    if contenido.get("version") != 1:
        raise ValueError("Versión de caché no reconocida.")

    return contenido["puntos"]


def escribir_cache_manual(registros):
    temporal = ARCHIVO_CACHE_MANUAL.with_suffix(".tmp")

    temporal.write_text(
        json.dumps(
            {"version": 1, "puntos": registros},
            ensure_ascii=False,
            indent=2,
            allow_nan=False,
        ),
        encoding="utf-8",
    )

    os.replace(temporal, ARCHIVO_CACHE_MANUAL)


def sincronizar_manuales():
    global catalogo_revision, catalogo_metodos

    filas = []

    for registro in leer_cache_manual():
        rid = str(registro["ID"])

        # Conservar en disco, pero no aplicar puntos a perfiles distintos.
        if (
            rid not in perfiles
            or registro["perfil_hash"] != huellas[rid]
        ):
            continue

        j = int(registro["indice"])
        p = perfiles[rid]

        if not 0 <= j < len(p):
            raise ValueError("Índice inválido en el caché manual.")

        tramos = [
            nombre
            for nombre, mascara in mascaras_tramos[rid].items()
            if np.asarray(mascara, dtype=bool)[j]
        ]

        if len(tramos) != 1:
            continue

        punto = p.iloc[j]
        chi = float(chi_sensibilidad[rid][THETA_DETECCION][j])

        if not np.isfinite([
            chi, punto["elevacion_m"],
            punto["x_utm"], punto["y_utm"],
        ]).all():
            continue

        filas.append({
            **registro,
            "ID": rid,
            "rio": nombres[rid],
            "metodo": "Manual",
            "tramo": tramos[0],
            "distancia_m": float(punto["distancia_m"]),
            "elevacion_m": float(punto["elevacion_m"]),
            "x_utm": float(punto["x_utm"]),
            "y_utm": float(punto["y_utm"]),
            "chi_m": chi,
            "theta": float(THETA_DETECCION),
            "me_interesa": True,
            "revision": "pendiente",
            "observaciones": "Selección manual mediante clic",
            "coincide_con": "",
            "estado_coincidencias": "no_evaluadas",
        })

    automaticos = catalogo_revision.loc[
        ~catalogo_revision["metodo"].eq("Manual")
    ].copy()

    catalogo_revision = (
        pd.concat([automaticos, pd.DataFrame(filas)], ignore_index=True)
        if filas else automaticos
    )

    catalogo_revision["ID"] = catalogo_revision["ID"].astype(str)

    if catalogo_revision["id_kp"].duplicated().any():
        raise ValueError("Hay IDs duplicados en el catálogo.")

    catalogo_metodos = catalogo_revision.copy()
    catalogo_metodos["etiqueta"] = catalogo_metodos["id_kp"]


# Incorporar los manuales a la función gráfica de tu sección 9.
ESTILOS["Manual"] = {
    "color": "black",
    "marker": "*",
    "s": 150,
}

if "Manual" not in METODOS_VER:
    METODOS_VER.append("Manual")

sincronizar_manuales()

ids_manuales_antes = {
    registro["id_kp"]
    for registro in leer_cache_manual()
}
    
try:
    for rid in map(str, RIOS_MANUALES):
        if rid not in perfiles:
            raise ValueError(f"No existe el río {rid}.")

        p = perfiles[rid]
        chi = np.asarray(
            chi_sensibilidad[rid][THETA_DETECCION],
            dtype=float,
        )
        z = p["elevacion_m"].to_numpy(float)

        fig, ax = plt.subplots(figsize=(12, 6))
        fig.canvas.manager.set_window_title(
            f"Selección manual — {nombres[rid]}"
        )

        def dibujar():
            ax.clear()
            ax.plot(chi, z, color="steelblue", linewidth=1.3)

            candidatos = catalogo_revision[
                catalogo_revision["ID"].eq(rid)
            ]

            for metodo, grupo in candidatos.groupby("metodo"):
                estilo = ESTILOS.get(
                    metodo,
                    {"color": "gray", "marker": "o", "s": 40},
                )

                ax.scatter(
                    grupo["chi_m"],
                    grupo["elevacion_m"],
                    color=estilo["color"],
                    marker=estilo["marker"],
                    s=estilo["s"],
                    label=metodo,
                    zorder=5,
                )

                if metodo == "Manual":
                    for _, candidato in grupo.iterrows():
                        ax.annotate(
                            candidato["id_kp"],
                            (
                                candidato["chi_m"],
                                candidato["elevacion_m"],
                            ),
                            xytext=(5, 8),
                            textcoords="offset points",
                            fontsize=8,
                        )

            ax.set(
                xlabel=r"$\chi$ (m)",
                ylabel="Elevación (m)",
                title=f"{rid} — {nombres[rid]} | Selección manual",
            )
            ax.grid(alpha=0.25)

            if not candidatos.empty:
                ax.legend()

            fig.tight_layout()
            fig.canvas.draw_idle()

        dibujar()
        plt.show(block=False)
        plt.pause(0.2)

        try:
            while plt.fignum_exists(fig.number):
                respuesta = input(
                    f"\n{nombres[rid]} — "
                    "¿Quieres agregar un knickpoint manual? [y/n]: "
                ).strip().lower()

                if respuesta == "n":
                    break

                if respuesta != "y":
                    print("Escribe y o n.")
                    continue

                print(
                    "Ve a la ventana del gráfico y haz un clic sobre el perfil.\n"
                    "Desactiva el botón de zoom/desplazamiento antes del clic.\n"
                    "Enter en la ventana cancela esta selección."
                )

                clics = fig.ginput(
                    n=1,
                    timeout=0,
                    show_clicks=True,
                )

                if not clics:
                    print("Selección cancelada.")
                    continue

                x_clic, _ = clics[0]
                validos = np.flatnonzero(
                    np.isfinite(chi) & np.isfinite(z)
                )

                if not len(validos):
                    print("El perfil no tiene nodos válidos.")
                    continue

                if not chi[validos].min() <= x_clic <= chi[validos].max():
                    print("El clic está fuera del perfil.")
                    continue

                # Ajustar al nodo más cercano en χ.
                j = int(validos[
                    np.argmin(np.abs(chi[validos] - x_clic))
                ])

                tramos = [
                    nombre
                    for nombre, mascara in mascaras_tramos[rid].items()
                    if np.asarray(mascara, dtype=bool)[j]
                ]

                if len(tramos) != 1:
                    print("Elige un nodo dentro de un único tramo de análisis.")
                    continue

                punto = p.iloc[j]

                if not np.isfinite(
                    punto[["x_utm", "y_utm"]].to_numpy(float)
                ).all():
                    print("Ese nodo tiene coordenadas inválidas.")
                    continue

                id_manual = (
                    f"R{int(rid):02d}_M_I{j:05d}_{huellas[rid][:8]}"
                )

                registros = leer_cache_manual()

                if any(r["id_kp"] == id_manual for r in registros):
                    print(f"El punto {id_manual} ya estaba guardado.")
                    continue

                registro = {
                    "id_kp": id_manual,
                    "ID": rid,
                    "indice": j,
                    "tramo": tramos[0],
                    "perfil_hash": huellas[rid],
                    "fecha_seleccion": datetime.now(
                        timezone.utc
                    ).isoformat(),
                    "vista_seleccion": "chi",
                    "posicion_clic": float(x_clic),
                    "theta_seleccion": float(THETA_DETECCION),
                    "A0_seleccion_m2": float(A0_ANALISIS),
                    "distancia_m": float(punto["distancia_m"]),
                    "elevacion_m": float(punto["elevacion_m"]),
                    "x_utm": float(punto["x_utm"]),
                    "y_utm": float(punto["y_utm"]),
                }

                escribir_cache_manual(registros + [registro])
                sincronizar_manuales()

                # Conservar el zoom mientras se añade el punto.
                limites_x, limites_y = ax.get_xlim(), ax.get_ylim()
                dibujar()
                ax.set_xlim(limites_x)
                ax.set_ylim(limites_y)
                plt.pause(0.1)

                print(
                    f"Guardado: {id_manual}\n"
                    f"χ = {chi[j]:.2f} m | "
                    f"Distancia = {punto['distancia_m'] / 1000:.3f} km | "
                    f"Elevación = {z[j]:.2f} m"
                )

        finally:
            plt.close(fig)

finally:
    # Volver a gráficos normales para la sección 9.
    get_ipython().run_line_magic("matplotlib", "inline")
from IPython.display import display

# Seleccionar únicamente los puntos agregados en esta ejecución.
nuevos = catalogo_revision.loc[
    catalogo_revision["metodo"].eq("Manual")
    & ~catalogo_revision["id_kp"].isin(ids_manuales_antes),
    [
        "ID", "rio", "id_kp",
        "distancia_m", "elevacion_m", "x_utm", "y_utm",
    ],
].copy()

nuevos = nuevos.sort_values(["rio", "distancia_m"])

tabla_nuevos = nuevos.rename(columns={
    "ID": "ID río",
    "rio": "Río",
    "id_kp": "ID knickpoint",
    "distancia_m": "Distancia (m)",
    "elevacion_m": "Elevación (m)",
    "x_utm": "Este UTM (m)",
    "y_utm": "Norte UTM (m)",
}).reset_index(drop=True)

print(f"\nKnickpoints nuevos agregados: {len(tabla_nuevos)}")
print("Coordenadas: WGS 84 / UTM zona 18S — EPSG:32718")

if tabla_nuevos.empty:
    print("No se agregaron puntos nuevos en esta ejecución.")
else:
    display(
        tabla_nuevos.style
        .format({
            "Distancia (m)": "{:.2f}",
            "Elevación (m)": "{:.2f}",
            "Este UTM (m)": "{:.2f}",
            "Norte UTM (m)": "{:.2f}",
        })
        .hide(axis="index")
    )
print(f"\nCaché guardado en:\n{ARCHIVO_CACHE_MANUAL}")
print("Ahora ejecuta la sección 9 y después la exportación.")

## 9. Visualización de candidatos seleccionados

Mostramos los IDs y destacamos con estrellas los candidatos de interés.

In [ ]:
# ── MODIFICAR AQUÍ ────────────────────────────────────────────────
RIOS_VER = list(perfiles)
# RIOS_VER = ["1", "3", "16"]
# RIOS_VER = list(perfiles)

MOSTRAR_ETIQUETAS = True

# Actualizar el catálogo que utiliza tu función gráfica anterior.
catalogo_metodos = catalogo_revision.copy()
catalogo_metodos["etiqueta"] = catalogo_metodos["id_kp"]

for river_id in map(str, RIOS_VER):
    fig = figura_revision(river_id)

    favoritos = catalogo_revision[
        catalogo_revision["ID"].eq(river_id)
        & catalogo_revision["me_interesa"]
        & catalogo_revision["metodo"].isin(METODOS_VER)
    ]

    for _, candidato in favoritos.iterrows():
        puntos = [
            (
                candidato["distancia_m"] / 1000,
                candidato["elevacion_m"],
            ),
            (candidato["chi_m"], candidato["elevacion_m"]),
            (candidato["x_utm"], candidato["y_utm"]),
        ]

        for ax, (x, y) in zip(fig.axes[:3], puntos):
            ax.scatter(
                x, y,
                marker="*",
                s=240,
                facecolors="none",
                edgecolors="black",
                linewidths=1.3,
                zorder=20,
            )

    if not favoritos.empty:
        fig.text(
            0.99, 0.98,
            "☆ Candidato de interés",
            ha="right", va="top", fontsize=9,
        )

    plt.show()
    plt.close(fig)

# Después de cambiar IDS_ME_GUSTAN, vuelve a ejecutar
# desde la celda del catálogo para actualizar selección y figuras.

## 10. Exportación del catálogo

Guardamos candidatos, selección personal, coincidencias y parámetros.
Opcionalmente generamos una capa de puntos para QGIS.